# Part 3 - Modelling

## Part 3 - Modelling
Target no_show. Baseline predict-all-attend = 78.62%. Split 75/25 stratified random_state=42 -> train 715, test 239. Features: age, gender, distance_km, lead_days, reminder_sent, clinic_id, fee_inr, booking_channel, visit_type, prior_no_shows. Numeric StandardScaler, categorical OneHot.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score, f1_score
df=pd.read_csv('appointments_clean.csv')
X=df[['age','gender','distance_km','lead_days','reminder_sent','clinic_id','fee_inr','booking_channel','visit_type','prior_no_shows']]; y=df['no_show']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
num=['age','distance_km','lead_days','fee_inr','prior_no_shows']; cat=['gender','reminder_sent','clinic_id','booking_channel','visit_type']
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
models={'logistic':Pipeline([('pre',pre),('clf',LogisticRegression(max_iter=1000))]),'tree_depth3':Pipeline([('pre',pre),('clf',DecisionTreeClassifier(max_depth=3,random_state=42))]),'tree_full':Pipeline([('pre',pre),('clf',DecisionTreeClassifier(random_state=42))]),'forest':Pipeline([('pre',pre),('clf',RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1))])}
for n,m in models.items():
    m.fit(Xtr,ytr)
    for s,Xs,ys in [('train',Xtr,ytr),('test',Xte,yte)]: print(n,s,round(accuracy_score(ys,m.predict(Xs)),4),round(roc_auc_score(ys,m.predict_proba(Xs)[:,1]),4))

### Results
Logistic train 0.8308/0.8152 test 0.7741/0.7311. Tree depth-3 train 0.8252/0.7449 test 0.7992/0.6033. Tree full train 1.0/1.0 test 0.7280/0.5628 - BELOW baseline 0.7862, memorised noise. Forest train 1.0/1.0 test 0.7782/0.7065. Forest importances top: lead_days, distance_km, age, fee_inr, prior_no_shows.
### Logistic interpretation
reminder_sent_Yes coef -0.423, No +0.423, difference -0.846, OR No->Yes = exp(-0.846) = 0.43. Odds ~57% lower with reminder, holding others. Association NOT causation.
### Tuning / chosen
GridSearch logistic C [0.1,1,10], 5-fold CV AUC: best C=0.1, CV AUC 0.792. CV logistic 0.787, tree-depth3 0.664, forest-tuned max_depth=8 0.755. Chosen = logistic (highest CV AUC, interpretable).
### Threshold 0.35
At 0.50: precision 0.44, recall 0.14, F1 0.21, flagged 16. At 0.35: precision 0.50, recall 0.37, F1 0.43, flagged 38/239. Confusion [[169,19],[32,19]].

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=42)
grid=GridSearchCV(Pipeline([('pre',pre),('clf',LogisticRegression(max_iter=1000))]),{'clf__C':[0.1,1.0,10.0]},cv=cv,scoring='roc_auc'); grid.fit(Xtr,ytr); print(grid.best_params_, round(grid.best_score_,4))
best=grid.best_estimator_; probs=best.predict_proba(Xte)[:,1]
for thr in [0.5,0.35]:
    p=(probs>=thr).astype(int); print(thr, round(precision_score(yte,p),3), round(recall_score(yte,p),3), round(f1_score(yte,p),3), int((p==1).sum())); print(confusion_matrix(yte,p))
t=Xte.copy(); t['no_show']=yte.values; t['risk_score']=probs; t['flag']=(probs>=0.35).astype(int); t['appointment_id']=df.loc[Xte.index,'appointment_id'].values; t.to_csv('test_risk_scores.csv',index=False); print(len(t), int(t.flag.sum()))